# Importing Libraries and Setting Up API

This cell prepares the environment for the MoE router:
- Import required libraries (`groq`, `python-dotenv`).
- Load environment variables from `.env`.
- Validate `GROQ_API_KEY` is available.
- Initialize the Groq client.

Install dependencies once (terminal):
`pip install groq python-dotenv`

In [12]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key from .env
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")

client = Groq(api_key=api_key)

# Defining Experts (Mixture of Experts Setup)

We define experts via system prompts while using the same Groq base model.

Experts used in this assignment:
- **Technical Expert**: rigorous, code-focused, precise debugging help.
- **Billing Expert**: empathetic, policy-driven support for charges/refunds.
- **General Expert**: fallback assistant for all other requests.

Bonus route:
- **Tool Use**: for real-time-like queries (mocked Bitcoin price lookup).

In [13]:
BASE_MODEL = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": (
            "You are a technical support expert. Be rigorous, precise, and code-focused. "
            "Diagnose root causes, suggest clear fixes, and provide concise code examples when useful."
        )
    },
    "billing": {
        "system_prompt": (
            "You are a billing support expert. Be empathetic and policy-driven. "
            "Clarify charges, refunds, subscriptions, and next steps in simple language."
        )
    },
    "general": {
        "system_prompt": (
            "You are a helpful general assistant for customer support. "
            "Be clear, polite, and practical."
        )
    }
}

VALID_CATEGORIES = {"technical", "billing", "general", "tool_use"}

# Router Function (Intent Classification)

The `route_prompt()` function classifies user input into exactly one category:
- `technical`
- `billing`
- `general`
- `tool_use` (bonus)

Constraints followed:
- Uses `temperature=0` for deterministic behavior.
- Prompt explicitly instructs the model to return **only** a category name.
- Adds a safety fallback to `general` if output is unexpected.

In [14]:
def route_prompt(user_input: str) -> str:
    routing_prompt = f"""
You are an intent classifier for customer support.
Classify the user's message into exactly one category:
technical, billing, general, tool_use.

Use tool_use only when user asks for live/external data (example: current Bitcoin price).
Return ONLY the category name in lowercase. No punctuation. No explanation.

User message: {user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,
        messages=[{"role": "user", "content": routing_prompt}]
    )

    category = response.choices[0].message.content.strip().lower()
    if category not in VALID_CATEGORIES:
        return "general"
    return category

# Orchestrator Function

The `process_request()` workflow:
1. Routes the query with `route_prompt()`.
2. If category is `tool_use`, calls a mock tool function directly.
3. Otherwise picks the corresponding expert system prompt.
4. Calls Groq with the selected expert prompt and user input.
5. Returns the final response text.

In [15]:
def get_bitcoin_price_mock() -> str:
    # Mock tool output for the bonus challenge.
    return "Mock data: Current Bitcoin price is $67,420.15 USD."


def process_request(user_input: str) -> str:
    category = route_prompt(user_input)
    print("Routed to:", category)

    if category == "tool_use":
        return get_bitcoin_price_mock()

    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])["system_prompt"]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content.strip()

In [16]:
test_queries = [
    "My Python script throws IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "What is the current price of Bitcoin?"
]

for q in test_queries:
    print("\nUser:", q)
    print("AI:", process_request(q))


User: My Python script throws IndexError on line 5.
Routed to: technical
AI: To assist you better, I'll need more information about your script. Please provide the following:

1. The exact error message you're seeing, including the line number.
2. The relevant code snippet that's causing the error.
3. Any relevant context or variables involved.

That being said, here are some general tips to diagnose and fix an `IndexError` in Python:

1. **Check the indexing**: Make sure you're not trying to access an index that's out of bounds. Python uses 0-based indexing, so the first element is at index 0.
2. **Verify array/list length**: Ensure that the array or list you're accessing has at least the number of elements you're trying to access.
3. **Check for empty lists**: If you're trying to access an element in an empty list, you'll get an `IndexError`.

Here's a simple example of how to diagnose and fix an `IndexError`:

**Code snippet**
```python
my_list = [1, 2, 3]
try:
    print(my_list[10

# Quick Test Cases

Run these examples once before the interactive loop:
- Technical query
- Billing query
- Bonus tool-use query

# Running the Smart Router

Some notebook kernels do not handle endless `input()` loops smoothly.
Use one of these options:
- `ask_once("your question")` for single queries (recommended in notebooks).
- `interactive_chat()` for multi-turn chat (bounded and safe to exit).

Type `exit` or `quit` in interactive mode to stop.

In [27]:
def ask_once(user_query: str) -> str:
    user_query = user_query.strip()
    if not user_query:
        return "Please enter a query."
    return process_request(user_query)


def interactive_chat(max_turns: int = 20) -> None:
    print("Type 'exit' or 'quit' to stop.")
    for _ in range(max_turns):
        try:
            user_query = input("Ask: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if user_query.lower() in {"exit", "quit"}:
            print("Session ended.")
            break

        if not user_query:
            print("Please enter a query.")
            continue

        answer = process_request(user_query)
        print("AI:", answer)
    else:
        print(f"Reached max_turns={max_turns}. Run interactive_chat() again to continue.")

# Notebook-friendly input (single query)
query = input("Enter your query: ").strip()
print("AI:", ask_once(query))

# Optional multi-turn mode:
# interactive_chat()

Routed to: billing
AI: I'm so sorry to hear that you were charged twice for your subscription this month. I'm here to help you resolve the issue. 

Let's break it down step by step. Can you please provide me with the following information: 

1. Your subscription plan and type (e.g., monthly, annual, individual, family)?
2. The date(s) you were charged twice?
3. The amount(s) you were charged each time?
4. Your account login information (email address or username associated with your subscription)?

Once I have this information, I'll be able to investigate the issue and provide you with a solution. 

As a policy-driven billing support expert, I want to assure you that we have a process in place to prevent duplicate charges. However, sometimes technical glitches can occur. I'm here to help you get your account corrected and ensure that you're not charged again for the duplicate transaction.

Please know that refunds will be processed as soon as possible if we find that you were indeed ch

# Conclusion

This assignment now includes a complete MoE routing pipeline:
- Expert configuration via specialized system prompts.
- Deterministic LLM-based intent router.
- Orchestrator that dispatches to the right expert.
- Correct use of routing temperature (`0`) and expert temperature (`0.7`).

Bonus implemented:
- A `tool_use` route that triggers a mock Bitcoin price function instead of an LLM response.